In [1]:
%pip install pandas numpy scikit-learn

In [2]:

import pandas as pd
import numpy as np

heart_data = pd.DataFrame({
    "chest_pain": [
        1, 1, 0, 1, 0, 0, 1, 0
    ],
    "blocked_arteries": [
        1, 0, 0, 1, 0, 1, 1, 0
    ],
    "body_weight": [
        85, 78, 60, 90, 65, 72, 88, 58
    ],
    "heart_patient": [
        1, 1, 0, 1, 0, 0, 1, 0
    ]
})

print(heart_data)


   chest_pain  blocked_arteries  body_weight  heart_patient
0           1                 1           85              1
1           1                 0           78              1
2           0                 0           60              0
3           1                 1           90              1
4           0                 0           65              0
5           0                 1           72              0
6           1                 1           88              1
7           0                 0           58              0


In [3]:
print("Rows and columns:", heart_data.shape)
print("Missing values:\n", heart_data.isnull().sum())
print("Target distribution:\n",
      heart_data["heart_patient"].value_counts())

Rows and columns: (8, 4)
Missing values:
 chest_pain          0
blocked_arteries    0
body_weight         0
heart_patient       0
dtype: int64
Target distribution:
 heart_patient
1    4
0    4
Name: count, dtype: int64


In [4]:

N = len(heart_data)

sample_weights = np.ones(N) / N

heart_data["initial_weight"] = sample_weights

print(heart_data[
    ["heart_patient", "initial_weight"]
])

print("Total weight:", sample_weights.sum())


   heart_patient  initial_weight
0              1           0.125
1              1           0.125
2              0           0.125
3              1           0.125
4              0           0.125
5              0           0.125
6              1           0.125
7              0           0.125
Total weight: 1.0


In [5]:
from sklearn.tree import DecisionTreeClassifier

X = heart_data[
    ["chest_pain", "blocked_arteries", "body_weight"]
]

y = heart_data["heart_patient"]

stump = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

stump.fit(X, y, sample_weight=sample_weights)

predictions = stump.predict(X)

results = pd.DataFrame({
    "actual": y,
    "predicted": predictions,
    "correct": y.to_numpy() == predictions
})

print(results)

   actual  predicted  correct
0       1          1     True
1       1          1     True
2       0          0     True
3       1          1     True
4       0          0     True
5       0          0     True
6       1          1     True
7       0          0     True


In [6]:

incorrect = (predictions != y.to_numpy())

weighted_error = np.sum(
    sample_weights * incorrect
) / np.sum(sample_weights)

print("Incorrect predictions:", incorrect.sum())
print("Weighted error:", weighted_error)


Incorrect predictions: 0
Weighted error: 0.0


In [7]:

if weighted_error >= 0.5:
    print(
        "This stump does not perform better than "
        "chance under the current weighting."
    )
else:
    alpha = 0.5 * np.log(
        (1 - weighted_error) / weighted_error
    )

    print("Weighted error:", weighted_error)
    print("Classifier contribution:", alpha)

if weighted_error < 0.5:
    updated_weights = sample_weights.copy()

    updated_weights[~incorrect] *= np.exp(-alpha)
    updated_weights[incorrect] *= np.exp(alpha)

    normalized_weights = (
        updated_weights / updated_weights.sum()
    )

    weight_results = pd.DataFrame({
        "actual": y.to_numpy(),
        "predicted": predictions,
        "correct": ~incorrect,
        "old_weight": sample_weights,
        "updated_weight": updated_weights,
        "normalized_weight": normalized_weights
    })

    print(weight_results.round(4))
    print(
        "Normalized total:",
        normalized_weights.sum()
    )


Weighted error: 0.0
Classifier contribution: inf


/tmp/ipykernel_3454/2034465719.py:8: RuntimeWarning: divide by zero encountered in scalar divide
  (1 - weighted_error) / weighted_error


In [8]:

if weighted_error < 0.5:
    updated_weights = sample_weights.copy()

    updated_weights[~incorrect] *= np.exp(-alpha)
    updated_weights[incorrect] *= np.exp(alpha)

    normalized_weights = (
        updated_weights / updated_weights.sum()
    )

    weight_results = pd.DataFrame({
        "actual": y.to_numpy(),
        "predicted": predictions,
        "correct": ~incorrect,
        "old_weight": sample_weights,
        "updated_weight": updated_weights,
        "normalized_weight": normalized_weights
    })

    print(weight_results.round(4))
    print(
        "Normalized total:",
        normalized_weights.sum()
    )

   actual  predicted  correct  old_weight  updated_weight  normalized_weight
0       1          1     True       0.125             0.0                NaN
1       1          1     True       0.125             0.0                NaN
2       0          0     True       0.125             0.0                NaN
3       1          1     True       0.125             0.0                NaN
4       0          0     True       0.125             0.0                NaN
5       0          0     True       0.125             0.0                NaN
6       1          1     True       0.125             0.0                NaN
7       0          0     True       0.125             0.0                NaN
Normalized total: nan


/tmp/ipykernel_3454/656413503.py:8: RuntimeWarning: invalid value encountered in divide
  updated_weights / updated_weights.sum()


In [9]:
y = pd.Series(
    [1, 0, 0, 1, 0, 1, 1, 0],
    name="heart_patient"
)

sample_weights = np.ones(len(y)) / len(y)

In [10]:

from sklearn.tree import DecisionTreeClassifier
import numpy as np
import pandas as pd

X = heart_data[
    ["chest_pain", "blocked_arteries", "body_weight"]
].reset_index(drop=True)

y = y.reset_index(drop=True)

weights = np.ones(len(y)) / len(y)

learners = []
alphas = []

for round_no in range(1, 6):
    tree = DecisionTreeClassifier(
        max_depth=1,
        random_state=round_no
    )

    tree.fit(X, y, sample_weight=weights)
    pred = tree.predict(X)

    incorrect = pred != y.to_numpy()

    error = np.sum(weights * incorrect)

    print(f"\nRound {round_no}")
    print("Weighted error:", round(error, 4))

    if error >= 0.5:
        print("Stopping: weak learner error is >= 0.5")
        break

    if error == 0:
        print("Perfect learner found; stopping.")
        learners.append(tree)
        alphas.append(1.0)
        break

    alpha = 0.5 * np.log((1 - error) / error)

    weights *= np.exp(
        alpha * np.where(incorrect, 1, -1)
    )
    weights /= weights.sum()

    learners.append(tree)
    alphas.append(alpha)

    print("Alpha:", round(alpha, 4))
    print("Updated weights:", np.round(weights, 4))

print("\nNumber of learners:", len(learners))



Round 1
Weighted error: 0.0
Perfect learner found; stopping.

Number of learners: 1


In [11]:

if not learners:
    print("No valid weak learners were trained.")
else:
    # Convert each stump's predictions to -1 or +1
    signed_predictions = np.array([
        2 * tree.predict(X) - 1
        for tree in learners
    ])

    # Weighted sum of all stump predictions
    final_scores = np.dot(
        np.array(alphas),
        signed_predictions
    )

    final_predictions = (
        final_scores >= 0
    ).astype(int)

    output = pd.DataFrame({
        "actual_label": y,
        "ensemble_score": final_scores,
        "predicted_label": final_predictions
    })

    print(output)


   actual_label  ensemble_score  predicted_label
0             1             1.0                1
1             0            -1.0                0
2             0            -1.0                0
3             1             1.0                1
4             0            -1.0                0
5             1             1.0                1
6             1             1.0                1
7             0            -1.0                0
